In [1]:
import os

keys = [
    "SNOWFLAKE_ACCOUNT",
    "SNOWFLAKE_USER",
    "SNOWFLAKE_PASSWORD",
    "SNOWFLAKE_DATABASE",
    "SNOWFLAKE_SCHEMA_RAW",
    "SNOWFLAKE_SCHEMA_ANALYTICS",
    "SNOWFLAKE_WAREHOUSE",
    "SNOWFLAKE_ROLE",
]

for k in keys:
    value = os.getenv(k)
    if value:
        if "PASSWORD" in k:
            print(f"{k}: ********")
        else:
            print(f"{k}: {value}")
    else:
        print(f"{k}: MISSING")

SNOWFLAKE_ACCOUNT: ynvpgpx-snc01960
SNOWFLAKE_USER: LEO
SNOWFLAKE_PASSWORD: ********
SNOWFLAKE_DATABASE: NYC_TLC
SNOWFLAKE_SCHEMA_RAW: RAW
SNOWFLAKE_SCHEMA_ANALYTICS: ANALYTICS
SNOWFLAKE_WAREHOUSE: COMPUTE_WH
SNOWFLAKE_ROLE: SYSADMIN


In [2]:
!pip install snowflake-connector-python
!pip install -U "pyarrow>=14.0.1"

In [3]:
import os
import snowflake.connector

conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    role=os.getenv("SNOWFLAKE_ROLE"),
)

cur = conn.cursor()

cur.execute(f"USE DATABASE {os.getenv('SNOWFLAKE_DATABASE')}")
cur.execute(f"USE SCHEMA {os.getenv('SNOWFLAKE_SCHEMA_RAW')}")

cur.execute("""
SELECT 
    CURRENT_VERSION(), 
    CURRENT_DATABASE(), 
    CURRENT_SCHEMA(), 
    CURRENT_WAREHOUSE(), 
    CURRENT_ROLE()
""")

print(cur.fetchone())

cur.close()
conn.close()

('10.11.2', 'NYC_TLC', 'RAW', 'COMPUTE_WH', 'SYSADMIN')


In [4]:
from pyspark.sql import SparkSession
import os

spark = (
    SparkSession.builder
    .appName("Snowflake Spark Test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "net.snowflake:spark-snowflake_2.12:2.12.0-spark_3.4",
            "net.snowflake:snowflake-jdbc:3.14.4"
        ])
    )
    .getOrCreate()
)

sfOptions = {
    "sfURL": f"{os.getenv('SNOWFLAKE_ACCOUNT')}.snowflakecomputing.com",
    "sfUser": os.getenv("SNOWFLAKE_USER"),
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD"),
    "sfDatabase": os.getenv("SNOWFLAKE_DATABASE"),
    "sfSchema": os.getenv("SNOWFLAKE_SCHEMA_RAW"),
    "sfWarehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "sfRole": os.getenv("SNOWFLAKE_ROLE"),
}

df = spark.read \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("query", "SELECT CURRENT_VERSION() AS version") \
    .load()

df.show()

+-------+
|VERSION|
+-------+
|10.11.2|
+-------+

